# 第3章 早高峰抽样调查

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch03-bootstrap-v1`  
**必做：** 日级简单随机抽样；百分位Bootstrap区间  
**对象：** 07、08、09时平均交通量：辆/小时  
**样本：** 2017年358个三小时完整日期；另有7日不完整  
**划分：** 固定完整日期抽样框；不推广为全部道路或未来年份  
**比较：** 主比较n=20/60/90，B=1000，seed=42；单独改变日期类型或种子

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 核查日期总体
请解释：358个完整日期与365个日历日期的差别，为什么不能将三个小时当作三个调查日？


In [ ]:
daily = pd.read_csv(ROOT/'chapters/data/ch03_daily.csv')
assert len(daily)==358 and daily.date.is_unique
display(daily.groupby('day_type').peak_mean.agg(['count','mean']))
print('Excluded dates:', sorted(set(pd.date_range('2017-01-01','2017-12-31').strftime('%Y-%m-%d'))-set(daily.date)))


## 2. 参数与随机抽样
主协议固定B=1000、seed=42。若改种子或日期类型，将其作为另一组实验记录，不能将不同总体成绩混用。


In [ ]:
SAMPLE_SIZES = [20,60,90]
B = 1000
SEED = 42
GROUP = 'all'  # all / weekday / weekend
pool = daily if GROUP=='all' else daily[daily.day_type==GROUP]
assert B>=50 and all(2<=n<=len(pool) for n in SAMPLE_SIZES)
# 与网页相同的LCG和Fisher-Yates顺序，便于逐项复核。
def random_stream(seed):
    state = seed
    while True:
        state = (1664525*state + 1013904223) % 2**32
        yield state/2**32
def sample_and_bootstrap(n):
    rnd = random_stream(SEED)
    order = list(range(len(pool)))
    for i in range(len(order)-1,0,-1):
        j=int(next(rnd)*(i+1));order[i],order[j]=order[j],order[i]
    sample=pool.iloc[order[:n]]
    values=sample.peak_mean.to_numpy(float)
    boot=np.array([np.mean([values[int(next(rnd)*n)] for _ in range(n)]) for _ in range(B)])
    return sample,boot


## 3. 形成区间并比较调查工作量
阅读并修改重采样代码。请解释：B增加与调查日期数增加分别改变什么？


In [ ]:
rows=[]
for n_days in SAMPLE_SIZES:
    sample,boot=sample_and_bootstrap(n_days)
    lower,upper=np.quantile(boot,[.025,.975])
    rows.append([sample.peak_mean.mean(),lower,upper,n_days,(upper-lower)/2])
    csv_file(ROOT/f'outputs/ch03/sample_{n_days}.csv',sample.columns,sample.values)
result=pd.DataFrame(rows,columns=['estimate','lower','upper','n_days','half_width'])
display(result)
csv_file(ROOT/'outputs/ch03/interval_comparison.csv',result.columns,result.values)
csv_file(ROOT/'outputs/ch03/ch03_submission.csv',result.columns[:4],[result.iloc[1,:4]])
plt.figure(figsize=(8,4));plt.errorbar(result.n_days,result.estimate,yerr=[result.estimate-result.lower,result.upper-result.estimate],fmt='o',capsize=5)
plt.xlabel('Survey days');plt.ylabel('Peak mean, vehicles/hour');save_plot(ROOT,3,'survey_precision')


## 4. 提出有条件的调查建议
不能将较窄区间解释为已经消除非随机缺失、日期相关和道路代表性问题。


In [ ]:
report(ROOT,3,'早高峰抽样调查建议',{'参数':{'seed':SEED,'B':B,'group':GROUP},'区间':result.to_string(index=False)},['选多少调查日，依据是什么？','完整日期筛选可能带来什么偏差？','对未来日期和其他断面能否外推？'])
